In [ ]:
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from skimage.io import imread

from edt import edt
from nd2 import ND2File


def get_pixel_size(file_path):
    with ND2File(file_path) as reader:
        # invert xyz voxel size to zyx to match img array
        pixel_size = reader.voxel_size()[::-1]
    return pixel_size


def get_edts(mask, pixel_size):
    # do 3D EDT
    dt = edt(mask, anisotropy=pixel_size)

    # get size of whole image in units
    img_size = np.array(pixel_size) * np.array(mask.shape)

    # do xy-EDT for each plane in stack
    # NOTE: to avoid doing this in a loop, we just set a fake pixel size in z
    # corresponding to the image size in xy (larger of the 2), thus all paths leading to a pixel should be just in xy 
    fake_pixel_size_xy_dt = [img_size[1:].max()] + list(pixel_size[1:])
    dt_xy = edt(mask, anisotropy=fake_pixel_size_xy_dt)

    # get distance just in z via fake xy pixel sizes corresponding to maximal z extent (see above)
    fake_pixel_size_z_dt = [pixel_size[0], img_size[0], img_size[0]]
    dt_z = edt(mask, anisotropy=fake_pixel_size_z_dt)

    return dt, dt_xy, dt_z


def get_distance_table(mask, spot_df, pixel_size):

    # column names of (pixel) zyx coordinates in the spot detection table
    coord_column_names = ['z', 'y', 'x']

    # calculate EDTs
    dt, dt_xy, dt_z = get_edts(mask, pixel_size)
    
    # "chache" of sorted distances for each label
    dt_sorted = {}
    dt_sorted_xy = {}
    dt_sorted_z = {}

    # dict of lists to build DataFrame
    res_df = defaultdict(list)

    for ri, coords in spot_df[coord_column_names].iterrows():
        # integer coordinate tuple to index arrays
        c_idx = tuple(coords.astype(int))
        # label from mask
        label = mask[c_idx]

        # sort all distances of object with index label (except background) and store in cache
        if label != 0 and not label in dt_sorted:
            dt_sorted[label] = np.sort(dt[mask==label])
            dt_sorted_xy[label] = np.sort(dt_xy[mask==label])
            dt_sorted_z[label] = np.sort(dt_z[mask==label])

        # get distances from EDTs
        d = dt[c_idx]
        d_xy = dt_xy[c_idx]
        d_z = dt_z[c_idx]

        # get quantiles in EDTs (index at which d would be inserted / total size)
        q = np.searchsorted(dt_sorted[label], d) / dt_sorted[label].size if label != 0 else np.nan
        q_xy = np.searchsorted(dt_sorted_xy[label], d_xy) / dt_sorted_xy[label].size if label != 0 else np.nan
        q_z = np.searchsorted(dt_sorted_z[label], d_z) / dt_sorted_z[label].size if label != 0 else np.nan

        res_df['label'].append(label)

        res_df['d'].append(d)
        res_df['d_xy'].append(d_xy)
        res_df['d_z'].append(d_z)

        res_df['q'].append(q)
        res_df['q_xy'].append(q_xy)
        res_df['q_z'].append(q_z)

    res_df = pd.DataFrame(res_df)
    return res_df

In [ ]:
base_path = Path('/data/agl_data/NanoFISH/Gabi/GS072_20230818_K562-EVI1-GFP_EVI-MYC/')

mask_path = base_path / 'segmentation-threshold'

spot_detection_path = base_path / 'spot-detection'

spot_feature_path = base_path / 'spot-mask-distances'

discard_spots_outside_mask = True


In [ ]:
mask_files = sorted(mask_path.glob('*_segmented.tif'))

spot_files = sorted(spot_detection_path.glob('*_spot-detection.csv'))

# check pairs of files
list(zip(mask_files, spot_files))

In [ ]:
# make out path if necessary
if not spot_feature_path.exists():
    spot_feature_path.mkdir(parents=True)


for mask_file, spot_file in zip(mask_files, spot_files):

    # get location of raw file (parent directory of segmentation + change back to nd2)
    raw_file = mask_path.parent / mask_file.name.replace('_segmented.tif', '.nd2')
    pixel_size = get_pixel_size(raw_file)

    # load mask
    mask = imread(mask_file)

    # load spot detection results
    spot_df = pd.read_csv(spot_file)

    res_df = get_distance_table(mask, spot_df, pixel_size)
    res_df['spot_detection_file'] = spot_file
    res_df['mask_file'] = mask_file
    res_df['channel'] = spot_df['channel']

    # add the index in the spot detection table as extra column 
    # -> this way, we can join later, even if we discard those outside mask
    res_df = res_df.reset_index()
    res_df = res_df.rename(columns={"index": "spot_detection_idx"})
    
    if discard_spots_outside_mask and len(res_df) > 0:
        res_df = (res_df[res_df.label != 0])

    out_file_path = spot_feature_path / spot_file.name.replace('_spot-detection.csv', '_spot-mask-distances.csv')
    res_df.to_csv(out_file_path, index=None)

    print(f'finished processing {mask_file}.')

## testing code for edt

In [ ]:
from matplotlib import pyplot as plt

mask = np.zeros((5, 5), dtype=int)
mask[1:, 1:] = 1

dt = edt(mask, anisotropy=(1, 1.5), black_border=False, order='F')
plt.imshow(dt)
dt